# HiveMind Agentic Benchmark: 4-Harness Arena (Core Suite & Polyglot Python)

Runtime: **A100 GPU** (40 GB oder 80 GB), High-RAM, >=100 GB Disk.

## Das Test-Setup (Harness vs. Harness auf identischem Modell)
Wir vergleichen nicht Modelle, sondern **Agenten-Harnesses**:
1. **`hivemind_duo`**: HiveMind Agentic Harness (Planner, Coder, Critic, AST-Guards, Auto-Lint, Fuzzy Diff, Multi-Turn Tool Loop)
2. **`aider`**: Aider CLI Harness (Diff Engine, Repo-Map)
3. **`hermes`**: Nous Research Hermes Agent CLI (`NousResearch/hermes-agent`)
4. **`pi`**: Mario Zechner Pi Coding Agent CLI (`earendil-works/pi`)

**Parität & Wissenschaftliche Sauberkeit:**
- **Identisches Modell & Gewichte:** Qwen3.6-35B-A3B (UD-Q4_K_XL) oder Hermes3.6-35B, voll auf die GPU geladen (76 GB VRAM-Budget, 0 MoE CPU Experts).
- **Gleiche Ressourcen:** Gleicher `llama-server`, identisches Modell, isolierte Workspaces, identisches Zeitlimit.
- **Kein Test-Leak:** Verdeckte Unit-Tests betreten niemals vor dem Check den Workspace. Volle Symmetrie.
- **Kein `raw`:** Rohe Zero-Shot-Diffs ohne Agenten-Tools messen nur das nackte Modell und verschwenden GPU-Stunden.

## Drei Phasen:
- **Phase 1: HiveMind Core Suite (23 Tasks, 32k Kontext)**: Multi-File, Refactorings, AST-Symbolerhalt, schwere Algorithmen (`hard_lru_cache`, `hard_cycle_detect`).
- **Phase 2: Polyglot Python Practice (34 Tasks, 32k Kontext)**: Der offizielle Python-Practice-Corpus aus `Aider-AI/polyglot-benchmark`.
- **Phase 3: DeepSWE Real-World Benchmark (Echte Repos, 128k Kontext)**: Reale Bugfixes aus `datacurve-ai/deep-swe` (`tomlkit`, `httpx`, `cattrs`) mit 131.072 Kontext-Tokens.


In [ ]:
# 1) GPU & Disk prüfen
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!df -h /root | tail -1


In [ ]:
# 2) Neuesten dev-Branch klonen / aktualisieren
![ -d /root/HiveMind_llama/.git ] || git clone --branch dev https://github.com/BoredLuzo/HiveMind_llama.git /root/HiveMind_llama
!cd /root/HiveMind_llama && git pull --ff-only
!cp /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh
!chmod +x /root/rent_bootstrap.sh
!cd /root/HiveMind_llama && git log -1 --format='%h %s'


In [ ]:
# 3) Setup (~15-20 min für 35B bzw. ~3 min für 4B)
# Installiert HiveMind, llama.cpp, Aider, Hermes & Pi
# Für 35B (Standard): !HM_LLAMA_CUDA=12.8 bash /root/rent_bootstrap.sh setup
# Für 4B (auf L4/T4/A100): !HM_MODEL="qwen3.5:4b-mtp" HM_LLAMA_CUDA=12.8 bash /root/rent_bootstrap.sh setup
!HM_LLAMA_CUDA=12.8 bash /root/rent_bootstrap.sh setup


In [ ]:
# 4) Engine starten (erkennt VRAM von A100, L4 oder T4 automatisch)
# Für 35B (Standard): !bash /root/rent_bootstrap.sh serve
# Für 4B mit 128k Ctx: !HM_MODEL="qwen3.5:4b-mtp" HM_CTX=131072 bash /root/rent_bootstrap.sh serve
!bash /root/rent_bootstrap.sh stop
!pkill -9 -f llama-server 2>/dev/null || true
!bash /root/rent_bootstrap.sh serve
!curl --max-time 5 -fsS http://127.0.0.1:8001/health


In [ ]:
# 4b) VRAM-Verifikation: Liegt das 35B-Modell komplett im VRAM (~22-26 GB)?
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader
!grep -E "VRAM-BUDGET|MoE experts|n-gpu-layers|n-cpu-moe|ubatch" /root/HiveMind_llama/bench_engine.log | tail -10


In [ ]:
# 5) Smoke-Test (3 Tasks: HiveMind vs. Aider, ~10-15 min)
!bash /root/rent_bootstrap.sh smoke


---
## PHASE 1: HiveMind Core Suite (23 Tasks)
Testet Multi-File-Navigation, Refactoring, AST-Symbolerhalt und Selbstkorrektur.
Head-to-Head: HiveMind Duo vs. Aider CLI auf identischem 35B-Modell.


In [ ]:
# 6a) Phase 1 starten: Core Suite (23 Tasks)
# Modell: Qwen 3.6 35B | Standard: hivemind_duo,aider | 4-Harness: hivemind_duo,aider,hermes,pi
import os
os.environ['HM_MODEL'] = 'qwen3.6:35b-a3b-ud'
os.environ['HM_RUNS'] = '1'
os.environ['HM_CONDITIONS'] = os.environ.get('HM_CONDITIONS', 'hivemind_duo,aider')
os.environ['HM_TASKS'] = 'bench/tasks'
!bash /root/rent_bootstrap.sh full


In [ ]:
# 6b) Optional: Claude Code auf derselben Core Suite ausführen (braucht ANTHROPIC_API_KEY)
import os
if os.environ.get('ANTHROPIC_API_KEY'):
    !cd /root/HiveMind_llama && .venv/bin/python bench/external/run_external.py --harness claude --tasks bench/tasks
else:
    print('Kein ANTHROPIC_API_KEY gesetzt — Claude Code wird übersprungen.')


In [ ]:
# 6c) Live-Status der Phase 1 überwachen (einmalig oder als Live-Monitor)
# Einmaliger Status:
!bash /root/rent_bootstrap.sh status
# ODER Live-Dashboard (aktualisiert alle 10s; Interrupt stoppt nur die Anzeige):
# !cd /root/HiveMind_llama && .venv/bin/python bench/watch.py


---
## PHASE 2: Polyglot Python Practice Benchmark (34 Tasks)
Das offizielle Benchmark-Set von Aider AI (`Aider-AI/polyglot-benchmark/python/exercises/practice`).
Testet den kompletten 34-Aufgaben-Corpus im direkten Head-to-Head auf demselben llama-server (~2-2.5h Laufzeit).


In [ ]:
# 7a) Alle 34 Polyglot-Python-Practice-Aufgaben von Aider AI herunterladen und importieren
!HM_POLYGLOT_NAMES=all bash /root/rent_bootstrap.sh polyglot


In [ ]:
# 7b) Phase 2 starten: Polyglot Python Practice (34 Tasks)
# Modell: Qwen 3.6 35B | Standard: hivemind_duo,aider | 4-Harness: hivemind_duo,aider,hermes,pi
import os
os.environ['HM_MODEL'] = 'qwen3.6:35b-a3b-ud'
os.environ['HM_RUNS'] = '1'
os.environ['HM_CONDITIONS'] = os.environ.get('HM_CONDITIONS', 'hivemind_duo,aider')
os.environ['HM_TASKS'] = 'bench/tasks_polyglot'
!bash /root/rent_bootstrap.sh full


In [ ]:
# 7c) Optional: Claude Code auf dem vollen Aider-Benchmark ausführen
import os
if os.environ.get('ANTHROPIC_API_KEY'):
    !cd /root/HiveMind_llama && .venv/bin/python bench/external/run_external.py --harness claude --tasks bench/tasks_polyglot
else:
    print('Kein ANTHROPIC_API_KEY gesetzt — Claude Code wird übersprungen.')


In [ ]:
# 7d) Live-Status der Phase 2 überwachen (einmalig oder als Live-Monitor)
# Einmaliger Status:
!bash /root/rent_bootstrap.sh status
# ODER Live-Dashboard:
# !cd /root/HiveMind_llama && .venv/bin/python bench/watch.py


---
## PHASE 3: DeepSWE Real-World Benchmark (Echte Repositories, 128k Kontext)
Schwere, reale Software-Engineering-Aufgaben aus `datacurve-ai/deep-swe` (`tomlkit`, `httpx`, `cattrs`).
Hier wird die volle Kontextstärke (128k / 131.072 Tokens) über alle Harnesses (`hivemind_duo`, `aider`, `hermes`, `pi`) hinweg getestet.
VRAM-Bedarf bei 128k: ~33 GB VRAM (passt perfekt in die Colab A100 mit 40/80 GB).


In [ ]:
# 8a) DeepSWE Python-Tasks importieren (lädt Repos & isolierte Hidden Tests ohne Docker)
# Installiert auch Test-Abhängigkeiten wie hypothesis und trio für den Test-Runner
!pip install -q hypothesis trio attrs pytest pytest-asyncio anyio
!bash /root/rent_bootstrap.sh deepswe


In [ ]:
# 8b) Engine mit 128k Kontext neu starten (~1 min)
!bash /root/rent_bootstrap.sh stop
!pkill -9 -f llama-server 2>/dev/null || true
!HM_CTX=131072 bash /root/rent_bootstrap.sh serve
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader


In [ ]:
# 8c) Phase 3 starten: DeepSWE Real-World Tasks (128k Kontext)
# Modell: Qwen 3.6 35B | Standard: hivemind_duo,aider | 4-Harness: hivemind_duo,aider,hermes,pi
import os
os.environ['HM_MODEL'] = 'qwen3.6:35b-a3b-ud'
os.environ['HM_RUNS'] = '1'
os.environ['HM_CONDITIONS'] = os.environ.get('HM_CONDITIONS', 'hivemind_duo,aider')
os.environ['HM_TASKS'] = 'bench/tasks_deepswe'
os.environ['HM_CTX'] = '131072'
os.environ['HM_CELL_TIMEOUT'] = '1800'
!bash /root/rent_bootstrap.sh full


In [ ]:
# 8d) Live-Status der Phase 3 überwachen (einmalig oder als Live-Monitor)
# Einmaliger Status:
!bash /root/rent_bootstrap.sh status
# ODER Live-Dashboard:
# !cd /root/HiveMind_llama && .venv/bin/python bench/watch.py


---
## AUSWERTUNG & REPORT (Das vereinte Leaderboard)
Analysiert die Ergebnisse wissenschaftlich (Wilson 95% CI, McNemar-Test, Net-Token-Ratio vs. Aider).


In [ ]:
# 9) Unified Leaderboard generieren
!cd /root/HiveMind_llama && .venv/bin/python bench/analyze.py --by-model --results $(ls -1 /root/HiveMind_llama/bench/results/*.json /root/HiveMind_llama/bench/results/external/*.json 2>/dev/null)


In [ ]:
# 10) Alle JSON-Ergebnisse dauerhaft in Google Drive sichern
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/hivemind_bench
!cp -v /root/HiveMind_llama/bench/results/*.json /content/drive/MyDrive/hivemind_bench/ 2>/dev/null || true
!cp -v /root/HiveMind_llama/bench/results/external/*.json /content/drive/MyDrive/hivemind_bench/ 2>/dev/null || true
!cp -v /root/HiveMind_llama/bench_full.log /content/drive/MyDrive/hivemind_bench/ 2>/dev/null || true
print('Ergebnisse erfolgreich in Google Drive gesichert!')


In [ ]:
# 11) Server stoppen
!bash /root/rent_bootstrap.sh stop
